# Day 13 结构特征第一轮 valid 实验

本轮只在 official training set 内部划分出的 `train_inner / valid` 上验证结构特征信号，不使用 official test 做评估或反向选择。

目标是比较几个收窄后的结构特征组是否能在 valid 上降低 total cost、减少 FN，并观察 recall、F2 和 PR-AUC 的变化。

## 1. 为什么 Day 13 只看 valid

- Day 12 只是结构特征设计，没有训练模型。
- Day 13 用 valid 选择结构特征方案和阈值。
- official test 留到 Day 14 对前 1-2 个 valid 候选方案做最终观察。
- 本轮不做 GridSearch、SHAP、PCA、SVM，也不解释匿名字段的真实物理含义。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid
from scania_aps.features.structural_feature_design import load_structural_feature_config
from scania_aps.features.structural_features import MAIN_EXPERIMENT_GROUPS
from scania_aps.models.structural_feature_experiments import run_structural_feature_valid_experiments

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
structural_config = load_structural_feature_config(PROJECT_ROOT / "config" / "structural_features.yaml")

## 2. 读取数据并划分 train_inner / valid

这里会读取 official train/test，但 Day 13 不使用 official test 做任何评估或方案选择。

In [2]:
train_df, _official_test_df = load_train_test_with_target(cfg)
train_inner_df, valid_df = split_train_valid(train_df, cfg)

pd.DataFrame({
    "dataset": ["official_train", "train_inner", "valid"],
    "rows": [len(train_df), len(train_inner_df), len(valid_df)],
    "positive_count": [train_df["target"].sum(), train_inner_df["target"].sum(), valid_df["target"].sum()],
    "positive_rate": [train_df["target"].mean(), train_inner_df["target"].mean(), valid_df["target"].mean()],
})

,dataset,rows,positive_count,positive_rate
0,official_train,60000,1000,0.016667
1,train_inner,48000,800,0.016667
2,valid,12000,200,0.016667


## 3. Day 13 实验组

本轮不把 Day 12 设计表中的 60 个结构特征直接作为主结论，而是按结构信号来源分组验证。`structural_all` 只作为上限观察。

In [3]:
MAIN_EXPERIMENT_GROUPS

['baseline_median_all',
 'median_all_sample_missing_rate',
 'median_all_selected_missing_indicators_top30',
 'median_all_prefix_zero_rate',
 'median_all_structural_core',
 'median_all_structural_all']

## 4. 运行 valid-only 结构特征实验

每个实验组都使用 `median_all` 作为原始数值特征基线，再拼接对应结构特征；模型范围收窄为 XGBoost，不做调参。

In [4]:
results = run_structural_feature_valid_experiments(
    train_inner_df=train_inner_df,
    valid_df=valid_df,
    cfg=cfg,
    structural_config=structural_config,
    experiment_groups=MAIN_EXPERIMENT_GROUPS,
)

valid_threshold_metrics = results["valid_threshold_metrics"]
valid_best_summary = results["valid_best_summary"]
experiment_metadata = results["experiment_metadata"]

cfg.metrics_dir.mkdir(parents=True, exist_ok=True)
cfg.tables_dir.mkdir(parents=True, exist_ok=True)

valid_threshold_metrics.to_csv(cfg.metrics_dir / "day13_structural_feature_valid_threshold_metrics.csv", index=False)
valid_best_summary.to_csv(cfg.metrics_dir / "day13_structural_feature_valid_best_summary.csv", index=False)
experiment_metadata.to_csv(cfg.tables_dir / "day13_structural_feature_experiment_metadata.csv", index=False)

## 5. valid best summary

重点看 total cost、FN、recall、F2 和 PR-AUC，而不是 accuracy。

In [5]:
display_cols = [
    "strategy", "best_threshold", "precision", "recall", "f2",
    "average_precision", "fp", "fn", "total_cost", "n_structural_features"
]
valid_best_summary[display_cols].sort_values(["total_cost", "recall", "f2"], ascending=[True, False, False])

,strategy,best_threshold,precision,recall,f2,average_precision,fp,fn,total_cost,n_structural_features
0,median_all_structural_all,0.18,0.367925,0.975,0.733083,0.863543,335,5,5850,60
1,median_all_selected_missing_indicators_top30,0.30,0.430804,0.965,0.773237,0.860375,255,7,6050,30
2,median_all_prefix_zero_rate,0.09,0.293592,0.985,0.669613,0.862527,474,3,6240,5
3,median_all_sample_missing_rate,0.15,0.342105,0.975,0.711679,0.866000,375,5,6250,1
4,baseline_median_all,0.16,0.359259,0.970,0.723881,0.867239,346,6,6460,0
5,median_all_structural_core,0.16,0.354015,0.970,0.719585,0.870606,354,6,6540,36


## 6. 分组观察

- `sample_missing_rate`：验证样本整体缺失程度是否有信号。
- `selected_missing_indicators_top30`：验证 Day 10 发现的 pos/neg 缺失差异字段是否有增益。
- `prefix_zero_rate`：验证 Day 11 中 ag / ay / cn / az / cs 前缀组零值结构是否有信号。
- `structural_core`：组合核心结构特征。
- `structural_all`：上限观察，不作为主结论。

In [6]:
experiment_metadata[[
    "experiment_group", "n_original_features", "n_structural_features",
    "n_total_features", "selected_missing_indicator_columns", "prefix_zero_groups"
]]

,experiment_group,n_original_features,n_structural_features,n_total_features,selected_missing_indicator_columns,prefix_zero_groups
0,baseline_median_all,170,0,170,,
1,median_all_sample_missing_rate,170,1,171,,
2,median_all_selected_missing_indicators_top30,170,30,200,br_000|bq_000|bp_000|bo_000|bn_000|bm_000|di_0...,
3,median_all_prefix_zero_rate,170,5,175,,ag|ay|cn|az|cs
4,median_all_structural_core,170,36,206,br_000|bq_000|bp_000|bo_000|bn_000|bm_000|di_0...,ag|ay|cn|az|cs
5,median_all_structural_all,170,60,230,br_000|bq_000|bp_000|bo_000|bn_000|bm_000|di_0...,ag|ay|cn|az|cs


## 7. Day 13 小结

本轮实验只形成 valid 候选结论：如果某个结构特征组在 valid 上 total cost 更低，只能说明它值得进入 Day 14 official test 最终观察，不能写成最终方案。

需要重点比较：

1. `selected_missing_indicators_top30` 是否稳定降低 FN 和 total cost；
2. `prefix_zero_rate` 是否带来独立结构信号；
3. `structural_core` 是否接近 `structural_all`，如果接近，应优先选择更简单的核心结构特征；
4. `structural_all` 如果更好，也要警惕冗余和过拟合风险。